# 04 — Backbone Comparison & Multi-Class Material Classification

**Experiment A**: Compare Swin-T RSP, SSL4EO ViT-S/16, DOFA ViT-B/16 on binary waste/no-waste task.

**Experiment B**: 20-class (22 with rare grouping) material classification on waste-positive AerialWaste images.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10

## Section 1 — Backbone Comparison (Experiment A)

In [ ]:
# Load results
with open('../data/processed/backbone_comparison_results.json') as f:
    backbone_results = json.load(f)

df = pd.DataFrame(backbone_results)
df['test/f1_pct'] = df['test/f1'] * 100
df['test/acc_pct'] = df['test/acc'] * 100
df

In [ ]:
# Bar chart: F1 and Accuracy
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = ['#2196F3', '#4CAF50', '#FF9800']
labels = df['label'].tolist()

# F1
bars = axes[0].bar(labels, df['test/f1_pct'], color=colors)
axes[0].set_ylabel('Test F1 (%)')
axes[0].set_title('Binary Classification — F1 Score')
axes[0].set_ylim(60, 100)
for bar, val in zip(bars, df['test/f1_pct']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.1f}%', ha='center', fontsize=9)

# Accuracy
bars = axes[1].bar(labels, df['test/acc_pct'], color=colors)
axes[1].set_ylabel('Test Accuracy (%)')
axes[1].set_title('Binary Classification — Accuracy')
axes[1].set_ylim(60, 100)
for bar, val in zip(bars, df['test/acc_pct']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.1f}%', ha='center', fontsize=9)

fig.tight_layout()
fig.savefig('../data/processed/backbone_f1_acc_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Training curves from CSV logs
import glob

fig, ax = plt.subplots(1, 1, figsize=(10, 6))

for label, color, ls in [
    ('SwinT_RSP', '#2196F3', '-'),
    ('SSL4EO_ViTS16', '#4CAF50', '--'),
    ('DOFA_ViTB16', '#FF9800', '-.'),
]:
    log_dirs = sorted(glob.glob(f'../logs/{label}/version_*/metrics.csv'))
    if not log_dirs:
        continue
    all_f1 = []
    for log_path in log_dirs:
        metrics = pd.read_csv(log_path)
        val_f1 = metrics[metrics['val/f1'].notna()]['val/f1'].values
        all_f1.extend(val_f1)
    if all_f1:
        ax.plot(range(len(all_f1)), [x * 100 for x in all_f1],
                label=label, color=color, linestyle=ls, linewidth=2)

ax.set_xlabel('Epoch')
ax.set_ylabel('Validation F1 (%)')
ax.set_title('Training Curves — val/F1 by Epoch')
ax.axvline(x=10, color='gray', linestyle=':', alpha=0.5, label='Phase 1→2')
ax.legend()
fig.tight_layout()
fig.savefig('../data/processed/backbone_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Efficiency frontier: Inference time vs F1
fig, ax = plt.subplots(1, 1, figsize=(8, 6))

for i, (_, row) in enumerate(df.iterrows()):
    ax.scatter(row['infer_ms'], row['test/f1_pct'], color=colors[i],
               s=row['params_M'] * 3, zorder=5, edgecolors='black', linewidth=0.5)
    ax.annotate(row['label'], (row['infer_ms'], row['test/f1_pct']),
                textcoords='offset points', xytext=(10, 5), fontsize=9)

ax.set_xlabel('Inference Time (ms/batch, batch=32)')
ax.set_ylabel('Test F1 (%)')
ax.set_title('Efficiency Frontier\n(bubble size ∝ parameter count)')
fig.tight_layout()
fig.savefig('../data/processed/backbone_efficiency.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Takeaway
best_idx = df['test/f1'].idxmax()
best = df.loc[best_idx]
print(f"Best backbone: {best['label']} (F1={best['test/f1']:.2%}, Acc={best['test/acc']:.2%})")
print(f"Parameters: {best['params_M']:.1f}M, Inference: {best['infer_ms']:.1f}ms/batch")
print()
print("Summary table:")
display(df[['label', 'test/f1_pct', 'test/acc_pct', 'params_M', 'infer_ms', 'total_time_min', 'precision']]
        .rename(columns={'test/f1_pct': 'F1(%)', 'test/acc_pct': 'Acc(%)',
                         'params_M': 'Params(M)', 'infer_ms': 'Infer(ms)',
                         'total_time_min': 'Train(min)'}))

## Section 2 — 22-Class Material Classification (Experiment B)

In [ ]:
# Load multiclass results
with open('../data/processed/multiclass_results.json') as f:
    mc_results = json.load(f)

class_names = mc_results['class_names']
per_class_f1 = mc_results['per_class_f1']
print(f"Macro F1: {mc_results['test/f1_macro']:.2%}")
print(f"Weighted F1: {mc_results['test/f1_weighted']:.2%}")
print(f"Accuracy: {mc_results['test/acc']:.2%}")
print(f"Classes: {mc_results['num_classes']}, Rare group: {mc_results['rare_group']}")

In [ ]:
# Confusion matrix heatmap
from PIL import Image
cm_img = Image.open('../data/processed/confusion_matrix_22class.png')
fig, ax = plt.subplots(1, 1, figsize=(14, 12))
ax.imshow(cm_img)
ax.axis('off')
ax.set_title('20-Class Confusion Matrix')
plt.show()

In [ ]:
# Per-class F1 bar chart (sorted)
sorted_f1 = sorted(per_class_f1.items(), key=lambda x: x[1], reverse=True)
names, values = zip(*sorted_f1)

fig, ax = plt.subplots(1, 1, figsize=(12, 7))
cmap = plt.cm.RdYlGn
bar_colors = [cmap(v) for v in values]
bars = ax.barh(range(len(names)), [v * 100 for v in values], color=bar_colors)
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=9)
ax.set_xlabel('Per-Class F1 (%)')
ax.set_title(f'Per-Class F1 (Macro={mc_results["test/f1_macro"]:.2%})')
ax.axvline(x=mc_results['test/f1_macro'] * 100, color='gray', linestyle='--', alpha=0.5, label='Macro F1')
ax.legend()
ax.invert_yaxis()

for i, (bar, val) in enumerate(zip(bars, values)):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1%}', va='center', fontsize=8)

fig.tight_layout()
fig.savefig('../data/processed/multiclass_per_class_f1.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Group analysis by coarse category
COARSE_GROUPS = {
    'organic': ['Fire Wood', 'Sludge-Zootechnical waste-Manure'],
    'construction': ['Rubble/excavated earth and rocks',
                     'Corrugated sheets (presumed asbestos-cement)',
                     'Stone/marble processing waste', 'Asphalt milling', 'Foundry waste'],
    'metal_plastic': ['Scrap', 'Big bags', 'Cisterns', 'Drums bins',
                      'Tires', 'Plastic'],
    'bulky': ['Bulky items', 'Vehicles', 'Domestic appliances',
              'Paper', 'Glass', 'Full pallets'],
    'storage': ['Heaps not delimited', 'Full container',
                'Delimited heaps (by barriers/walls/etc)'],
}

group_f1 = {}
for group, members in COARSE_GROUPS.items():
    matched = [per_class_f1.get(m, 0) for m in members if m in per_class_f1]
    # Also check Rare for members that were grouped
    rare_members = mc_results.get('rare_group', {}).get('Rare', [])
    for m in members:
        if m in rare_members and 'Rare' in per_class_f1:
            matched.append(per_class_f1['Rare'])
            break
    if matched:
        group_f1[group] = np.mean(matched)

fig, ax = plt.subplots(1, 1, figsize=(8, 5))
grp_names = list(group_f1.keys())
grp_vals = [group_f1[g] * 100 for g in grp_names]
grp_colors = ['#66BB6A', '#42A5F5', '#AB47BC', '#FFA726', '#EF5350']
bars = ax.bar(grp_names, grp_vals, color=grp_colors[:len(grp_names)])
ax.set_ylabel('Mean F1 (%)')
ax.set_title('F1 by Coarse Category Group')
for bar, val in zip(bars, grp_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', fontsize=9)
fig.tight_layout()
fig.savefig('../data/processed/multiclass_group_f1.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Spectral hypothesis: which bands would help the worst-performing classes?
print("=== Spectral Discrimination Hypothesis ===")
print()
print("Bottom-3 classes by F1:")
for name, f1_val in sorted_f1[-3:]:
    print(f"  {name}: F1={f1_val:.2%}")

print()
print("From spectral_band_rationale.md:")
print("- Construction materials (concrete, rubble, brick): discriminated by")
print("  RED-EDGE (650-750nm) and NIR (850-1000nm) → S2 B5-B8a")
print("- Plastics, tires, polymer waste: diagnostic SWIR absorptions at")
print("  1210, 1415, 1660, 1728nm → requires WV3 SWIR")
print("- Metal/scrap: characteristically low NIR reflectance → S2 B8")
print("- Organic (wood): strong NIR reflection contrast vs minerals → S2 B8")
print()
print("→ The classes with worst RGB-only F1 are predicted to benefit")
print("  most from multispectral input. This motivates the MS extension.")